In [ ]:
!nvidia-smi

Thu Apr 23 19:54:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

/content


In [ ]:
# Pip install method (recommended)

!pip install --upgrade ultralytics
from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.9/235.7 GB disk)


In [ ]:
from ultralytics import YOLO

from IPython.display import display, Image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Zip dosyasının yolu (Örneğin Kernel'de bulunan: '/content/drive/MyDrive/data (1).zip')
zip_path = "/content/drive/MyDrive/Sayzek/dataset_augmented_yolo_23.04.2026.zip"

# Dosyaların çıkarılacağı hedef klasör
extract_dir = "/content/drive/MyDrive/Sayzek/dataset_augmented_yolo"

# Klasörü oluştur ve zip'ten çıkar (-q parametresi sessizce çıkarmak içindir)
!mkdir -p "{extract_dir}"
!unzip -q "{zip_path}" -d "{extract_dir}"

print(f"{zip_path} başarıyla {extract_dir} klasörüne çıkarıldı!")

/content/drive/MyDrive/Sayzek/dataset_augmented_yolo_23.04.2026.zip başarıyla /content/drive/MyDrive/Sayzek/dataset_augmented_yolo klasörüne çıkarıldı!


In [ ]:
%cd /content/drive/MyDrive/Sayzek

!yolo task=detect mode=train model=yolov8n.pt data=/content/drive/MyDrive/Sayzek/dataset_augmented_yolo/dataset_augmented_yolo/data.yaml epochs=100 imgsz=640 plots=True batch=32 save=True

/content/drive/MyDrive/Sayzek
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Sayzek/dataset_augmented_yolo/dataset_augmented_yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, 

In [ ]:
%%writefile cb_loss.py
"""
cb_loss.py — Class-Balanced Loss Kütüphanesi
=============================================
Cui et al. (2019) "Class-Balanced Loss Based on Effective Number of Samples"
https://arxiv.org/abs/1901.05555

KULLANIM — Temel (classification):
    from cb_loss import ClassBalancedCrossEntropyLoss, ClassBalancedFocalLoss

    criterion = ClassBalancedCrossEntropyLoss(
        class_counts = [50000, 10000, 3000, 800],
        beta         = 0.9999,
    )
    loss = criterion(logits, targets)   # logits: (B,C)  targets: (B,)

KULLANIM — Object detection (anchor/pred bazlı):
    from cb_loss import ClassBalancedDetectionClsLoss

    cls_loss_fn = ClassBalancedDetectionClsLoss(
        class_counts    = [50000, 10000, 3000, 800],
        beta            = 0.9999,
        background_idx  = None,   # arka plan sınıf indeksi (varsa)
        loss_type       = "focal", # "ce" | "focal" | "bce"
    )

    # Anchor-based / tek aşamalı (YOLO, NanoDet, RetinaNet):
    loss = cls_loss_fn(pred_logits, assigned_labels)
    # pred_logits    : (N, C)  — tüm pozitif anchor tahminleri
    # assigned_labels: (N,)    — her anchor'un atandığı sınıf

    # İki aşamalı (Faster R-CNN ROI head):
    loss = cls_loss_fn(roi_logits, roi_labels)
    # roi_logits : (R, C+1)  — ROI classifier çıktıları
    # roi_labels : (R,)      — her ROI'nin etiket indeksi

BETA KILAVUZU
-------------
beta, veri örtüşme derecesini temsil eder (0 < β < 1).

    0.9    : Ağırlıklar birbirine çok yaklaşır. Sınıflar arası fark
             neredeyse kaybolur. Çok fazla augmentation uygulanmış
             veri setlerinde kullanılır.

    0.99   : Orta düzey örtüşme. Dengeli bir başlangıç noktasıdır.

    0.999  : Düşük-orta örtüşme. Çoğu gerçek dünya veri setinde
             iyi bir başlangıç noktasıdır.

    0.9999 : Düşük örtüşme. Sınıflar arası ağırlık farkı belirgindir.
             Ham ve çeşitli veri setleri için önerilir. (varsayılan)

    Kural: Augmentation arttıkça beta'yı düşür,
           veri çeşitliliği arttıkça beta'yı yükselt.

GAMMA KILAVUZU (focal loss için)
----------------------------------
gamma, focal loss'un "zor örnek odaklanma" gücünü belirler (γ ≥ 0).

    0.0 : Focal terimi devre dışı. CB ağırlıklı CE'ye eşdeğer.
    1.0 : Hafif odaklanma.
    2.0 : Standart değer (RetinaNet varsayılanı). Önerilen başlangıç.
    5.0 : Güçlü odaklanma. Çok zor ya da gürøltølü veri setlerinde
          modeli kararsızlaşırabilir, dikkatli kullanın.
"""

import json as _json
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Optional


# ──────────────────────────────────────────────────────────────
# COCO'dan Otomatik Sınıf Sayısı Hesaplama
# ──────────────────────────────────────────────────────────────

def compute_class_counts_from_coco(
    annotation_path: str,
) -> tuple[list[str], list[int]]:
    with open(annotation_path, "r") as f:
        coco = _json.load(f)

    sorted_cats = sorted(coco["categories"], key=lambda x: x["id"])
    class_names = [cat["name"] for cat in sorted_cats]

    cat_counts = Counter(ann["category_id"] for ann in coco["annotations"])
    class_counts = [cat_counts.get(cat["id"], 0) for cat in sorted_cats]

    return class_names, class_counts


# ──────────────────────────────────────────────────────────────
# Çekirdek: Ağırlık Hesaplama
# ──────────────────────────────────────────────────────────────

def compute_cb_weights(
    class_counts: list[int],
    beta: float = 0.9999,
    normalize: bool = True,
    device: Optional[torch.device | str] = None,
) -> torch.Tensor:
    if not (0 < beta < 1):
        raise ValueError(f"beta (0, 1) aralığında olmalı, alınan: {beta}")
    if any(n <= 0 for n in class_counts):
        raise ValueError("Tüm class_counts değerleri pozitif tam sayı olmalı.")

    counts = np.array(class_counts, dtype=np.float64)
    effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
    weights = 1.0 / effective_num

    if normalize:
        weights = weights / weights.sum() * len(class_counts)

    tensor = torch.tensor(weights, dtype=torch.float32)
    if device is not None:
        tensor = tensor.to(device)
    return tensor


# ──────────────────────────────────────────────────────────────
# Loss 1: Classification (standart kullanım)
# ──────────────────────────────────────────────────────────────

class ClassBalancedCrossEntropyLoss(nn.Module):
    def __init__(
        self,
        class_counts: list[int],
        beta: float = 0.9999,
        label_smoothing: float = 0.0,
        reduction: str = "mean",
        device: Optional[torch.device | str] = None,
    ):
        super().__init__()
        weights = compute_cb_weights(class_counts, beta=beta, device=device)
        self.loss_fn = nn.CrossEntropyLoss(
            weight=weights,
            label_smoothing=label_smoothing,
            reduction=reduction,
        )

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return self.loss_fn(logits, targets)


class ClassBalancedFocalLoss(nn.Module):
    def __init__(
        self,
        class_counts: list[int],
        beta: float = 0.9999,
        gamma: float = 2.0,
        reduction: str = "mean",
        device: Optional[torch.device | str] = None,
    ):
        super().__init__()
        if gamma < 0:
            raise ValueError(f"gamma >= 0 olmalı, alınan: {gamma}")
        self.gamma = gamma
        self.reduction = reduction
        self.register_buffer(
            "weights",
            compute_cb_weights(class_counts, beta=beta, device=device)
        )

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.softmax(logits, dim=1)
        p_t = probs[torch.arange(len(targets), device=logits.device), targets]
        alpha_t = self.weights[targets]
        loss = -alpha_t * (1.0 - p_t) ** self.gamma * torch.log(p_t + 1e-8)

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# ──────────────────────────────────────────────────────────────
# Loss 2: Detection — model-agnostic cls loss
# ──────────────────────────────────────────────────────────────

class ClassBalancedDetectionClsLoss(nn.Module):
    def __init__(
        self,
        class_counts: list[int],
        beta: float = 0.9999,
        gamma: float = 2.0,
        loss_type: str = "focal",
        background_idx: Optional[int] = None,
        reduction: str = "mean",
        device: Optional[torch.device | str] = None,
    ):
        super().__init__()

        if loss_type not in ("focal", "ce", "bce"):
            raise ValueError(f"loss_type 'focal', 'ce' veya 'bce' olmalı, alınan: {loss_type}")
        if gamma < 0:
            raise ValueError(f"gamma >= 0 olmalı, alınan: {gamma}")

        self.loss_type = loss_type
        self.gamma = gamma
        self.reduction = reduction
        self.background_idx = background_idx

        fg_weights = compute_cb_weights(class_counts, beta=beta, normalize=True)

        if background_idx is not None:
            num_classes = len(class_counts) + 1
            weights = torch.ones(num_classes, dtype=torch.float32)
            fg_indices = [i for i in range(num_classes) if i != background_idx]
            weights[fg_indices] = fg_weights
        else:
            weights = fg_weights

        self.register_buffer("weights", weights.to(device) if device else weights)

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        if self.weights.device != logits.device:
            self.weights = self.weights.to(logits.device)

        if self.background_idx is not None:
            fg_mask = targets != self.background_idx
            if fg_mask.sum() == 0:
                return logits.sum() * 0.0
            logits  = logits[fg_mask]
            targets = targets[fg_mask]

        if self.loss_type == "ce":
            return F.cross_entropy(
                logits, targets,
                weight=self.weights,
                reduction=self.reduction,
            )

        elif self.loss_type == "focal":
            probs = torch.softmax(logits, dim=1)
            p_t = probs[torch.arange(len(targets), device=logits.device), targets]
            alpha_t = self.weights[targets]
            loss = -alpha_t * (1.0 - p_t) ** self.gamma * torch.log(p_t + 1e-8)

        else:  # bce
            num_classes = logits.size(1)
            one_hot = torch.zeros_like(logits)
            one_hot.scatter_(1, targets.unsqueeze(1), 1)
            alpha_t = self.weights.unsqueeze(0).expand_as(logits)
            loss = F.binary_cross_entropy_with_logits(
                logits, one_hot,
                weight=alpha_t,
                reduction="none",
            ).sum(dim=1)

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


Writing cb_loss.py


In [ ]:
import yaml
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
from ultralytics.utils.loss import v8DetectionLoss
from cb_loss import compute_cb_weights

DATA_YAML = "/content/drive/MyDrive/Sayzek/dataset_augmented_yolo/dataset_augmented_yolo/data.yaml"
BETA = 0.9999

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg["names"]
nc = len(names) if isinstance(names, (list, dict)) else int(names)

train_rel = data_cfg.get("train")
train_path = train_rel if Path(train_rel).is_absolute() else str(Path(DATA_YAML).parent / train_rel)
labels_dir = Path(train_path.replace("/images", "/labels"))
if not labels_dir.exists():
    labels_dir = Path(train_path).parent / "labels"

counts = Counter()
for lbl in labels_dir.rglob("*.txt"):
    with open(lbl) as f:
        for line in f:
            parts = line.split()
            if parts:
                counts[int(parts[0])] += 1

class_counts = [max(counts.get(i, 0), 1) for i in range(nc)]
print("nc:", nc, "class_counts:", class_counts)

cb_w = compute_cb_weights(class_counts, beta=BETA)
print("cb_weights:", [round(x, 4) for x in cb_w.tolist()])

class CBBCE(nn.Module):
    def __init__(self, weights):
        super().__init__()
        self.register_buffer("w", weights.view(1, 1, -1))
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
    def forward(self, pred, target):
        return self.bce(pred, target) * self.w.to(pred.device)

_orig_init = v8DetectionLoss.__init__
def _patched_init(self, model, tal_topk=10):
    _orig_init(self, model, tal_topk=tal_topk)
    self.bce = CBBCE(cb_w.clone())
v8DetectionLoss.__init__ = _patched_init
print("v8DetectionLoss patched → CB-weighted BCE active")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=32,
    plots=True,
    save=True,
    project="/content/drive/MyDrive/Sayzek/runs_cb",
    name="train",
)


In [ ]:
# Load a model
model = YOLO('/content/drive/MyDrive/Sayzek/runs/detect/train/weights/last.pt')  # load a partially trained model

# Resume training
results = model.train(resume=True)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Sayzek/dataset_augmented_yolo/dataset_augmented_yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Sayzek/runs/detect/train/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.

KeyboardInterrupt: 

## Validate Custom Model

In [ ]:
%cd /content/drive/MyDrive/SaymadımZek_YuksleCokCiddi/Veri_Setleri/Models/YOLOv8n

!yolo task=detect mode=val model=/content/drive/MyDrive/SaymadımZek_YuksleCokCiddi/Veri_Setleri/Models/YOLOv8n/runs/detect/train/weights/best.pt data=/content/drive/MyDrive/SaymadımZek_YuksleCokCiddi/Veri_Setleri/Models/YOLOv8n/ata-simurg-2/data.yaml

/content/drive/MyDrive/SaymadımZek_YuksleCokCiddi/Veri_Setleri/Models/YOLOv8n
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.4 ms, read: 12.7±3.6 MB/s, size: 29.6 KB)
val: Scanning /content/drive/MyDrive/SaymadımZek_YuksleCokCiddi/Veri_Setleri/Models/YOLOv8n/ata-simurg-2/valid/labels.cache... 1442 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1442/1442 15.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 91/91 3.8it/s 23.9s
                   all       1442       1466      0.949      0.934      0.979      0.659
Speed: 1.7ms preprocess, 3.8ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/drive/MyDrive/SaymadımZek_YuksleCokCiddi/Veri_Setleri/Models/YOLOv8n/runs/detect/val
💡 Learn more at https://docs.ultralytics.com/modes/val
